# 02 · Retrieval and answering

Follows a question through the system: the router picks a path, hybrid retrieval finds events, and the answer comes back with its evidence, source mix and a map.

**Needs:** the search indexes (`uv run rag-ucdp index`, once). Section 4 also needs Ollama running with the answer model; everything else works without it.

In [ ]:
from rag_ucdp.config import DEFAULT_CONFIG_PATH, load_config
from rag_ucdp.pipeline import PATH_LABELS, answer, check_status, load_resources

cfg = load_config(DEFAULT_CONFIG_PATH)
for item in check_status(cfg):
    print(f"{'ok' if item.ok else '--'}  {item.name:<13} {item.detail}")

In [ ]:
res = load_resources(cfg)  # loads the embedding model; takes a moment the first time
print(f"{len(res.df):,} events · {res.vector_store.count:,} vectors · {len(res.bm25):,} BM25 documents")

## 1. Routing: which path does a question take?

The router is rule-based: keyword lists decide the path, and place and year mentions become a filter. Every decision can be traced to a rule (`reason`).

In [ ]:
from rag_ucdp.router import classify

questions = [
    "What kinds of attacks happened in Mariupol?",
    "How many fatalities in Donetsk oblast in 2023?",
    "How many attacks on energy infrastructure in 2023?",
    "Why did Russia invade Ukraine?",
]
for q in questions:
    d = classify(q)
    print(f"{PATH_LABELS[d.path]:<15} {q}")
    print(f"{'':<15} filter: {d.filters or '—'}")
    print(f"{'':<15} reason: {d.reason}\n")

Path C (counting over retrieved events) is switched off in `config.yaml`, because it counted a fixed number of retrieved events. Questions routed there are answered on the semantic path instead.

## 2. Why hybrid retrieval?

Dense search (BGE-large) matches meaning; BM25 matches exact words. Each fails where the other is strong, so the system runs both and fuses the rankings with Reciprocal Rank Fusion.

In [ ]:
def show(label, rows, k=3):
    print(f"  {label}")
    for r in rows[:k]:
        doc = r.get("document") or res.vector_store.get_by_ids([r["id"]])[r["id"]]["document"]
        print(f"    {r['id']:>7}  {doc.splitlines()[-1][:90]}")

for q in ["attacks on schools", "Kerch bridge"]:
    print(f"\nQuestion: {q!r}")
    dense = res.vector_store.query(res.embedder.embed([q])[0], top_k=3)
    keyword = res.bm25.query(q, top_k=3)
    show("dense (BGE-large):", dense)
    show("keyword (BM25):", keyword)

Compare the lists: a concept such as *"attacks on schools"* can be phrased many ways in a headline, which favours dense search; a name such as *"Kerch bridge"* is matched exactly by BM25. The hybrid retriever below combines both and applies the router's filter to each leg before ranking.

In [ ]:
from rag_ucdp.retrieval import hybrid_retrieve

q = "strikes on power infrastructure in winter"
results = hybrid_retrieve(q, res.embedder, res.vector_store, res.bm25, top_k=5,
                          where=classify(q).filters or None)
def rank(r):
    return "—" if r is None else r + 1

for r in results:
    print(f"{r.id:>7}  dense #{rank(r.vector_rank):<3} bm25 #{rank(r.bm25_rank):<3} "
          f"rrf {r.rrf_score:.4f}  {r.document.splitlines()[-1][:70]}")

## 3. A numeric question: exact aggregation, no LLM

Counts and sums come straight from the event table, so the number is exact and reproducible.

In [ ]:
a = answer("How many fatalities in Donetsk oblast in 2023?", res)
print(PATH_LABELS[a.path], "·", a.text)
print("\nSource mix:", a.source_bias.description)

## 4. A descriptive question: retrieval + cited LLM answer

The LLM sees only the top retrieved events and must cite them as `[event-ID]`. Without Ollama, the answer shows an error but the evidence is still retrieved.

In [ ]:
a = answer("What kinds of attacks happened in Mariupol?", res)
print(PATH_LABELS[a.path], "· filter:", a.decision.filters)
print()
print(a.text)
print("\nCited:", a.cited_ids or "—")
print("Source mix:", a.source_bias.description)

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {
            "event": r.id,
            "date": r.metadata.get("date_start"),
            "oblast": r.metadata.get("adm_1"),
            "in prompt": i < (a.prompt_top_k or 0),
            "cited": r.id in a.cited_ids,
        }
        for i, r in enumerate(a.events)
    ]
)

## 5. On the map

In [ ]:
from rag_ucdp.maps import make_event_map

make_event_map(a.events_df)

The same `answer()` function powers the web app (`uv run rag-ucdp app`) and the terminal (`uv run rag-ucdp ask "…"`).